In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)


In [ ]:
import pandas as pd

df=pd.read_csv(r"C:\Users\Pragn\Desktop\loan-default-project\data\credit_risk_dataset.csv")
df


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

In [ ]:
df['loan_status'].value_counts()

In [ ]:
import seaborn as sns
sns.boxplot(x=df['person_emp_length'])

In [ ]:
sns.boxplot(x=df['loan_int_rate'])

In [ ]:
df['person_emp_length'].fillna(df['person_emp_length'].median() , inplace=True) 
df['loan_int_rate'].fillna(df['loan_int_rate'].mean() , inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df.select_dtypes(include='object').columns

In [ ]:
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder() 
df['loan_grade'] = le.fit_transform(df['loan_grade']) 
df['cb_person_default_on_file'] = le.fit_transform(df['cb_person_default_on_file']) 
df=pd.get_dummies(df,columns=['person_home_ownership','loan_intent'],drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X=df.drop('loan_status', axis=1)
y=df['loan_status']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced',max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf=RandomForestClassifier(n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp)

# Plot
plt.figure(figsize=(10,6))
plt.barh(feat_imp['feature'], feat_imp['importance'])
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.show()


In [ ]:
from xgboost import XGBClassifier
xgb_model=XGBClassifier(n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,   # imbalance handle
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb_model = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_model))
print(confusion_matrix(y_test, y_pred_xgb_model))
print(classification_report(y_test, y_pred_xgb_model))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    xgb_model,
    X,   # features
    y,   # target (loan_status)
    cv=cv,
    scoring='accuracy'
)

cv_scores


In [ ]:
print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Std Dev:", cv_scores.std())


In [ ]:
from src.model import train_xgb_model

train_xgb_model(
    csv_path="data/loan.csv",
    model_path="models/xgb_model.pkl"
)


# Final whole code

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)
import pandas as pd

df=pd.read_csv(r"C:\Users\Pragn\Desktop\loan-default-project\data\credit_risk_dataset.csv")
df
df.info()
df.describe()
df.isna().sum()
df.nunique()
df['loan_status'].value_counts()
import seaborn as sns
sns.boxplot(x=df['person_emp_length'])
sns.boxplot(x=df['loan_int_rate'])
df['person_emp_length'].fillna(df['person_emp_length'].median() , inplace=True) 
df['loan_int_rate'].fillna(df['loan_int_rate'].mean() , inplace=True)
df.isna().sum()
df.select_dtypes(include='object').columns
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder() 
df['loan_grade'] = le.fit_transform(df['loan_grade']) 
df['cb_person_default_on_file'] = le.fit_transform(df['cb_person_default_on_file']) 
df=pd.get_dummies(df,columns=['person_home_ownership','loan_intent'],drop_first=True)
from sklearn.model_selection import train_test_split

X=df.drop('loan_status', axis=1)
y=df['loan_status']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced',max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
from sklearn.ensemble import RandomForestClassifier

rf=RandomForestClassifier(n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp)

# Plot
plt.figure(figsize=(10,6))
plt.barh(feat_imp['feature'], feat_imp['importance'])
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.show()
from xgboost import XGBClassifier
xgb_model=XGBClassifier(n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,   # imbalance handle
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb_model = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_model))
print(confusion_matrix(y_test, y_pred_xgb_model))
print(classification_report(y_test, y_pred_xgb_model))
from sklearn.model_selection import StratifiedKFold, cross_val_score
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    xgb_model,
    X,   # features
    y,   # target (loan_status)
    cv=cv,
    scoring='accuracy'
)

cv_scores
print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Std Dev:", cv_scores.std())
from src.model import train_xgb_model

train_xgb_model(
    csv_path="data/loan.csv",
    model_path="models/xgb_model.pkl"
)

# EDA code

In [2]:
import sys
import os
PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)
print("Project root added:", PROJECT_ROOT)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


Project root added: C:\Users\Pragn\Desktop\loan-default-project


In [ ]:
df = pd.read_csv(r"C:\Users\Pragn\Desktop\loan-default-project\data\credit_risk_dataset.csv")

df.info()
df.describe()
df.isna().sum()
df.nunique()
df['loan_status'].value_counts()


In [ ]:
sns.boxplot(x=df['person_emp_length'])
sns.boxplot(x=df['loan_int_rate'])


In [ ]:
# Optional (EDA learning)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# Feature importance plot (EDA purpose)
importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feat_imp['feature'], feat_imp['importance'])
plt.gca().invert_yaxis()
plt.show()


In [3]:
from src.model import train_xgb_model

train_xgb_model(
    csv_path="../data/credit_risk_dataset.csv"
    
)


Accuracy: 0.9369015467714216
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      6369
           1       0.97      0.73      0.83      1777

    accuracy                           0.94      8146
   macro avg       0.95      0.86      0.90      8146
weighted avg       0.94      0.94      0.93      8146

✅ Model saved at: C:\Users\Pragn\Desktop\loan-default-project\models\xgb_model.pkl
✅ Scaler saved at: C:\Users\Pragn\Desktop\loan-default-project\models\scaler.pkl
